# Laboratorio 2.4 — Calidad y gobierno de datos (apoyo en pandas)

Este notebook detecta y corrige, paso a paso, los problemas de calidad presentes en `tienda_online_ventas_dirty.csv`. Cada celda de detección imprime cuántos casos ha encontrado — **antes de ejecutar este notebook, ya deberías haber intentado encontrar tú mismo estos problemas** (actividad 1 del enunciado). Usa las salidas de aquí para confirmar lo que encontraste, no como primer contacto con el dataset.

**Instrucciones:**
1. Sube este notebook y `tienda_online_ventas_dirty.csv` a la misma sesión de Colab.
2. Ejecuta las celdas en orden.
3. Al final se genera `tienda_online_ventas_limpio.csv` en el mismo directorio — es parte de tu entregable.

## 0. Carga del dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("tienda_online_ventas_dirty.csv")
n_filas_inicial = len(df)
print(f"Dataset cargado: {n_filas_inicial} filas x {df.shape[1]} columnas")
df.head()

## 1. Detección de problemas

Antes de corregir nada, detectamos y contamos cada tipo de problema sobre el dataset tal cual llega.

### 1.1 Filas duplicadas

In [ ]:
n_duplicados_raw = df.duplicated().sum()
print(f"Filas duplicadas (comparación exacta, sin normalizar antes): {n_duplicados_raw}")

### 1.2 Nulos inesperados

In [ ]:
columnas_con_nulos_inesperados = ["producto_nombre", "ciudad", "categoria", "metodo_pago"]

print("Nulos por columna:")
print(df.isnull().sum())
print()
print("Nota: 'valoracion' también tiene muchos nulos, pero son legítimos (no todo el mundo "
      "valora su compra) — igual que en el laboratorio 2.1, no los tratamos como un problema "
      "de calidad y no los vamos a modificar.")

### 1.3 Inconsistencias de casing (mayúsculas/minúsculas)

In [ ]:
print("Valores únicos en 'categoria':")
print(df["categoria"].value_counts(dropna=False))
print()
print("Valores únicos en 'canal':")
print(df["canal"].value_counts(dropna=False))

Se puede apreciar que `categoria` mezcla valores como `Electrónica` y `ELECTRÓNICA` (misma categoría, distinto casing), y `canal` mezcla `Web` con `web`, `App móvil` con `app móvil`, etc. Contamos cuántas filas están afectadas:

In [ ]:
categorias_canonicas = ["Electrónica", "Hogar", "Deporte", "Moda", "Papelería"]
canales_canonicos = ["Web", "App móvil", "Marketplace"]

filas_categoria_no_canonica = df["categoria"].notna() & ~df["categoria"].isin(categorias_canonicas)
filas_canal_no_canonico = df["canal"].notna() & ~df["canal"].isin(canales_canonicos)

print(f"Filas con 'categoria' en formato no estándar: {filas_categoria_no_canonica.sum()}")
print(f"Filas con 'canal' en formato no estándar: {filas_canal_no_canonico.sum()}")

### 1.4 Valores fuera de rango: `cantidad`, `precio_unitario` y `fecha`

In [ ]:
print("Estadísticos de 'cantidad':")
print(df["cantidad"].describe())
print()
print("Estadísticos de 'precio_unitario':")
print(df["precio_unitario"].describe())

In [ ]:
mask_cantidad_invalida = (df["cantidad"] < 1) | (df["cantidad"] > 20)
mask_precio_invalido = df["precio_unitario"] <= 0
mask_fecha_futura = df["fecha"] == "2027-01-15"

print(f"Filas con 'cantidad' fuera de rango (fuera de 1-20, valor esperado de negocio 1-5 con margen): "
      f"{mask_cantidad_invalida.sum()}")
print(f"Filas con 'precio_unitario' negativo o cero: {mask_precio_invalido.sum()}")
print(f"Filas con 'fecha' futura imposible (2027-01-15): {mask_fecha_futura.sum()}")

### 1.5 `importe` inconsistente con `cantidad × precio_unitario`

In [ ]:
importe_esperado = (df["cantidad"] * df["precio_unitario"]).round(2)
mask_importe_inconsistente = (importe_esperado - df["importe"]).abs() > 0.01

print(f"Filas donde 'importe' no coincide con 'cantidad x precio_unitario': "
      f"{mask_importe_inconsistente.sum()}")

### 1.6 Espacios extra en `cliente_id`

In [ ]:
mask_cliente_id_con_espacios = df["cliente_id"].astype(str) != df["cliente_id"].astype(str).str.strip()
print(f"Filas con 'cliente_id' con espacios extra al principio o al final: "
      f"{mask_cliente_id_con_espacios.sum()}")

## 2. Limpieza

Aplicamos las correcciones en un orden concreto y por una razón: **normalizamos texto (casing, espacios) antes de buscar duplicados**, porque dos filas que en el fichero original parecían distintas por una simple diferencia de mayúsculas en realidad son la misma fila duplicada. Si buscásemos duplicados antes de normalizar, se nos escaparían algunos.

### 2.1 Normalizar espacios en `cliente_id`

In [ ]:
df["cliente_id"] = df["cliente_id"].astype(str).str.strip()
print("cliente_id normalizado (espacios eliminados).")

### 2.2 Normalizar casing en `categoria` y `canal`

In [ ]:
mapa_categoria = {c.lower(): c for c in categorias_canonicas}
mapa_canal = {c.lower(): c for c in canales_canonicos}

# .where(condicion, valor_si_falso) mantiene los NaN tal cual y solo transforma los valores no nulos
df["categoria"] = df["categoria"].where(
    df["categoria"].isnull(),
    df["categoria"].str.strip().str.lower().map(mapa_categoria),
)
df["canal"] = df["canal"].where(
    df["canal"].isnull(),
    df["canal"].str.strip().str.lower().map(mapa_canal),
)

print("Valores únicos en 'categoria' tras normalizar:")
print(df["categoria"].value_counts(dropna=False))
print()
print("Valores únicos en 'canal' tras normalizar:")
print(df["canal"].value_counts(dropna=False))

### 2.3 Eliminar duplicados exactos (ya con texto normalizado)

In [ ]:
n_duplicados_normalizado = df.duplicated().sum()
print(f"Duplicados exactos detectados tras normalizar texto: {n_duplicados_normalizado} "
      f"(frente a los {n_duplicados_raw} detectados antes de normalizar)")

df = df.drop_duplicates().reset_index(drop=True)
print(f"Filas tras eliminar duplicados: {len(df)}")

### 2.4 Eliminar fechas futuras imposibles

In [ ]:
filas_antes = len(df)
df = df[df["fecha"] != "2027-01-15"].reset_index(drop=True)
print(f"Filas eliminadas por fecha futura imposible: {filas_antes - len(df)}")
print(f"Filas restantes: {len(df)}")

### 2.5 Eliminar filas con `cantidad` o `precio_unitario` inválidos

Estos son valores evidentemente erróneos (una compra de 200-900 unidades de un producto en una tienda online, o un precio negativo o cero) que no se pueden corregir con confianza — no hay forma de saber cuál era el valor correcto, así que se descartan las filas afectadas en vez de inventar un valor.

In [ ]:
filas_antes = len(df)
df = df[(df["cantidad"] >= 1) & (df["cantidad"] <= 20)].reset_index(drop=True)
print(f"Filas eliminadas por 'cantidad' fuera de rango: {filas_antes - len(df)}")

filas_antes = len(df)
df = df[df["precio_unitario"] > 0].reset_index(drop=True)
print(f"Filas eliminadas por 'precio_unitario' inválido: {filas_antes - len(df)}")

print(f"Filas restantes: {len(df)}")

### 2.6 Recalcular `importe` inconsistente

A diferencia de `cantidad` y `precio_unitario` (que si son incorrectos no se pueden reconstruir), `importe` es un campo **derivado**: siempre debería ser `cantidad × precio_unitario`. Aquí sí podemos corregirlo con confianza, recalculándolo en vez de descartar la fila.

In [ ]:
importe_esperado = (df["cantidad"] * df["precio_unitario"]).round(2)
mask_importe_inconsistente = (importe_esperado - df["importe"]).abs() > 0.01

print(f"Filas con 'importe' inconsistente que se van a recalcular: {mask_importe_inconsistente.sum()}")

df.loc[mask_importe_inconsistente, "importe"] = importe_esperado[mask_importe_inconsistente]
print("'importe' recalculado donde era necesario.")

### 2.7 Gestionar nulos inesperados

Para `producto_nombre`, `ciudad`, `categoria` y `metodo_pago`, no hay forma de reconstruir el valor real que faltaba, pero tampoco queremos perder toda la fila (el resto de columnas —importe, fecha, cliente— siguen siendo información válida). Los sustituimos por la etiqueta explícita `"Desconocido"`, que dejará claro en cualquier análisis posterior que ese dato faltaba, en vez de un hueco silencioso.

`valoracion` se deja tal cual: como vimos en el laboratorio 2.1, sus nulos son una ausencia legítima (no todo el mundo valora su compra), no un error de calidad.

In [ ]:
for columna in columnas_con_nulos_inesperados:
    n_nulos = df[columna].isnull().sum()
    print(f"Nulos en '{columna}' antes de rellenar: {n_nulos}")
    df[columna] = df[columna].fillna("Desconocido")

print()
print(f"Nulos en 'valoracion' (se dejan sin modificar, son legítimos): {df['valoracion'].isnull().sum()}")

## 3. Verificación final

In [ ]:
print(f"Filas iniciales: {n_filas_inicial}")
print(f"Filas finales: {len(df)}")
print(f"Filas eliminadas en total: {n_filas_inicial - len(df)}")
print()
print(f"Duplicados restantes: {df.duplicated().sum()}")
print(f"Nulos restantes por columna:")
print(df.isnull().sum())
print()
print(f"cantidad fuera de rango restante: {((df['cantidad'] < 1) | (df['cantidad'] > 20)).sum()}")
print(f"precio_unitario <= 0 restante: {(df['precio_unitario'] <= 0).sum()}")
print(f"fechas futuras restantes: {(df['fecha'] == '2027-01-15').sum()}")
importe_esperado_final = (df["cantidad"] * df["precio_unitario"]).round(2)
print(f"'importe' inconsistente restante: {((importe_esperado_final - df['importe']).abs() > 0.01).sum()}")

### Por qué la celda anterior puede mostrar `Duplicados restantes: 10`

Si al ejecutar la celda de verificación ves `Duplicados restantes: 10` en lugar de 0, **no es un error tuyo ni un fallo del notebook** — es un efecto real de los datos que vale la pena entender, y es el motivo por el que en el paso 2.3 detectamos 307 duplicados pero al final quedan 10 sin eliminar.

**Qué ha pasado:** el dataset original contiene 5 pedidos que están duplicados dos veces cada uno (mismo `pedido_id`, mismo cliente, mismo producto...), pero con el campo `importe` corrompido de forma *distinta* en cada copia. En el paso 2.3 (`drop_duplicates()`), esas 10 filas todavía no eran idénticas — las distinguía justamente el `importe` incorrecto — así que no se detectaron como duplicadas ahí. Solo al llegar al paso 2.6, cuando recalculamos `importe = cantidad × precio_unitario` para corregirlo, ambas copias de cada pedido acaban con el mismo valor correcto... y en ese momento se vuelven filas 100% idénticas. Como el `drop_duplicates()` ya se había ejecutado antes (paso 2.3), esas 10 filas nunca se llegan a eliminar.

Puedes verlas tú mismo:

```python
df[df.duplicated(keep=False)].sort_values("pedido_id")
```

Verás 5 pares de filas (10 filas en total), cada par con el mismo `pedido_id` repetido.

**La lección de fondo:** la limpieza de datos casi nunca es una sola pasada — corregir un problema (`importe` inconsistente) puede dejar al descubierto otro que antes estaba oculto (duplicados "disfrazados" por un valor erróneo). Es exactamente el tipo de caso que conviene comentar en las preguntas de reflexión del enunciado. Si quisieras dejar el dataset final con cero duplicados, la forma correcta sería añadir una **segunda** llamada a `drop_duplicates()` después del paso 2.6 (no adelantar la del paso 2.3, porque en ese punto `importe` todavía no está corregido y perderíamos la trazabilidad de qué filas tenían el importe mal calculado).

Y sobre los nulos: que `valoracion` siga mostrando nulos (1.779 en esta ejecución) tras la limpieza también es lo esperado — es el único campo que este notebook nunca toca, porque son valoraciones que el cliente simplemente no dejó, no un error de calidad (ver la nota del paso 0).

In [ ]:
# Bonus: si quisieras un dataset final con CERO duplicados (incluidos los que
# solo se revelan después de corregir 'importe'), bastaría con una segunda
# pasada de drop_duplicates() aquí. La dejamos comentada a propósito: el
# notebook no la aplica por defecto, para que la comprobación de la celda
# anterior muestre el caso real y puedas razonarlo tú mismo.

# df = df.drop_duplicates().reset_index(drop=True)
# print(f"Filas tras la segunda pasada de deduplicación: {len(df)}")

## 4. Guardar el dataset limpio

In [ ]:
df.to_csv("tienda_online_ventas_limpio.csv", index=False)
print(f"Guardado 'tienda_online_ventas_limpio.csv' con {len(df)} filas.")

## Siguiente paso

Con el dataset limpio ya generado, vuelve al enunciado (`enunciado.md`, actividades 3 y 4) para completar la ficha de catálogo (`ficha-catalogo-plantilla.md`), incluida la tabla resumen de la limpieza aplicada — puedes copiar directamente las cifras impresas en la sección 3 de este notebook.